# Memoria en redes recurrentes: RNN, LSTM y GRU

Esta demo compara la capacidad de tres arquitecturas recurrentes para conservar información a lo largo de una secuencia. La tarea es deliberadamente simple: la red recibe una secuencia de dígitos, uno por paso de tiempo, y al final debe indicar cuál fue el **primer** dígito. El resto de la secuencia actúa como ruido que debe ignorarse.

Con secuencias cortas la tarea es trivial. Al aumentar la longitud, la red debe mantener el dato en su estado oculto durante muchos pasos, y es ahí donde se manifiestan las diferencias entre arquitecturas. Como hay diez dígitos posibles, una exactitud cercana al 10 % indica que la red no retuvo la información y responde al azar.

Los datos se generan de forma sintética, por lo que no es necesario descargar nada. Se recomienda ejecutar la libreta con GPU.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # evita un conflicto de OpenMP en algunas instalaciones locales

import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

K = 10          # número de dígitos posibles (0-9)
HIDDEN = 64     # tamaño del estado oculto
BATCH = 128

## Datos

Cada secuencia tiene longitud `T`. El primer elemento es el dígito que debe recordarse y los demás son dígitos aleatorios. Cada dígito se representa como un vector *one-hot* de dimensión 10.

In [ ]:
def generar_batch(T, n, generador):
    """Regresa x con forma (n, T, K) en one-hot, y el primer dígito de cada secuencia como etiqueta."""
    digitos = torch.randint(0, K, (n, T), generator=generador)
    x = nn.functional.one_hot(digitos, K).float()
    y = digitos[:, 0]
    return x.to(device), y.to(device), digitos

g = torch.Generator().manual_seed(0)
_, y, digitos = generar_batch(T=15, n=3, generador=g)
for d, etiqueta in zip(digitos, y):
    print("Secuencia:", d.tolist(), " -> etiqueta:", etiqueta.item())

## Modelo

Los tres modelos comparten la misma estructura: una capa recurrente que recorre la secuencia y una capa lineal que, a partir del último estado oculto, predice el primer dígito. Solo cambia el tipo de celda recurrente.

La LSTM y la GRU controlan su memoria mediante compuertas. En la LSTM, la compuerta de olvido determina qué fracción del estado anterior se conserva en cada paso; en la GRU, esa función la cumple la compuerta de actualización. El parámetro `sesgo_compuerta` permite inicializar el sesgo de dicha compuerta con un valor positivo, de modo que al inicio del entrenamiento la compuerta esté casi abierta y la red tienda a conservar su estado. La RNN simple carece de compuertas, por lo que este parámetro no la afecta.

In [ ]:
class RedRecurrente(nn.Module):
    def __init__(self, tipo, sesgo_compuerta=0.0):
        super().__init__()
        celda = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[tipo]
        self.rnn = celda(K, HIDDEN, batch_first=True)
        self.salida = nn.Linear(HIDDEN, K)
        if sesgo_compuerta and tipo in ("LSTM", "GRU"):
            # Orden de las compuertas en PyTorch: LSTM (i, f, g, o) y GRU (r, z, n).
            # En ambos casos el segundo bloque corresponde a la compuerta que regula la memoria.
            for nombre, p in self.rnn.named_parameters():
                if "bias_ih" in nombre:
                    p.data[HIDDEN:2 * HIDDEN] = sesgo_compuerta

    def forward(self, x):
        h, _ = self.rnn(x)               # h: (batch, T, HIDDEN)
        return self.salida(h[:, -1])     # se usa únicamente el último paso de tiempo

for tipo in ["RNN", "GRU", "LSTM"]:
    n = sum(p.numel() for p in RedRecurrente(tipo).parameters())
    print(f"{tipo:5s}: {n:,} parámetros")

## Entrenamiento

En cada paso se genera un lote nuevo de secuencias, de modo que la red nunca ve dos veces el mismo ejemplo. La exactitud se mide cada 100 pasos sobre un conjunto de validación fijo, y el entrenamiento se detiene si alcanza el 99 %.

In [ ]:
def entrenar(tipo, T, sesgo_compuerta=0.0, pasos=3000, lr=3e-3, semilla=0):
    torch.manual_seed(semilla)
    g_train = torch.Generator().manual_seed(1000 + semilla)
    x_val, y_val, _ = generar_batch(T, 2000, torch.Generator().manual_seed(999))

    modelo = RedRecurrente(tipo, sesgo_compuerta).to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    historia, t0 = [], time.time()

    for paso in range(1, pasos + 1):
        x, y, _ = generar_batch(T, BATCH, g_train)
        loss = nn.functional.cross_entropy(modelo(x), y)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()

        if paso % 100 == 0:
            with torch.no_grad():
                acc = (modelo(x_val).argmax(1) == y_val).float().mean().item()
            historia.append((paso, acc))
            if acc >= 0.99:
                break

    mejor = max(acc for _, acc in historia)
    nombre = tipo + (f" (sesgo={sesgo_compuerta:g})" if sesgo_compuerta else "")
    print(f"{nombre:18s} T={T:4d} | exactitud = {mejor:6.1%} | pasos = {paso:5d} | {time.time() - t0:5.1f} s")
    return historia, mejor


def graficar(curvas, titulo):
    plt.figure(figsize=(8, 4))
    for nombre, historia in curvas.items():
        pasos, accs = zip(*historia)
        plt.plot(pasos, accs, marker="o", ms=3, label=nombre)
    plt.axhline(0.1, ls="--", c="gray", lw=1, label="azar (10 %)")
    plt.ylim(0, 1.05)
    plt.xlabel("paso de entrenamiento")
    plt.ylabel("exactitud en validación")
    plt.title(titulo)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## Secuencias cortas

Con `T = 10` el dígito relevante está a solo diez pasos del final. Se entrenan los tres modelos con su configuración por defecto.

In [ ]:
curvas = {}
for tipo in ["RNN", "LSTM", "GRU"]:
    curvas[tipo], _ = entrenar(tipo, T=10)
graficar(curvas, "T = 10")

Las tres arquitecturas resuelven la tarea. Cuando la dependencia es corta, la RNN simple es suficiente.

## Secuencias largas

Con `T = 50` se comparan cuatro configuraciones: la RNN simple, la LSTM con su inicialización por defecto, y la LSTM y la GRU con la compuerta de memoria inicializada casi abierta (`sesgo_compuerta = 4`).

In [ ]:
T = 50
curvas = {}
curvas["RNN"], _ = entrenar("RNN", T)
curvas["LSTM"], _ = entrenar("LSTM", T)
curvas["LSTM (sesgo=4)"], _ = entrenar("LSTM", T, sesgo_compuerta=4.0)
curvas["GRU (sesgo=4)"], _ = entrenar("GRU", T, sesgo_compuerta=4.0)
graficar(curvas, "T = 50")

La RNN simple permanece en el nivel del azar: no logra relacionar la etiqueta con un dato observado cincuenta pasos antes. La LSTM con inicialización por defecto tampoco aprende, lo que muestra que disponer de compuertas no basta por sí solo. Con sesgo cero, la compuerta de olvido inicia en σ(0) = 0.5, y tras cincuenta pasos la contribución del primer dato se reduce en un factor del orden de 0.5⁵⁰ ≈ 10⁻¹⁵.

Al inicializar la compuerta casi abierta (σ(4) ≈ 0.98), la LSTM y la GRU conservan el dato desde el comienzo del entrenamiento y aprenden la tarea en pocos cientos de pasos. Este mecanismo, que permite mantener el estado de forma casi intacta a lo largo del tiempo, es precisamente lo que la RNN simple no posee.

## El gradiente a lo largo del tiempo

Para que la red aprenda a conservar el primer dígito, el error calculado al final de la secuencia debe propagarse hacia atrás hasta el primer paso. La siguiente gráfica muestra la norma del gradiente de la pérdida respecto a la entrada en cada paso de tiempo, ‖∂L/∂xₜ‖, para modelos recién inicializados con `T = 100`.

In [ ]:
def gradiente_por_paso(modelo, T=100, n=256):
    x, y, _ = generar_batch(T, n, torch.Generator().manual_seed(7))
    x.requires_grad_(True)
    loss = nn.functional.cross_entropy(modelo(x), y)
    loss.backward()
    return x.grad.norm(dim=2).mean(0).detach().cpu().numpy()   # una norma por paso de tiempo

torch.manual_seed(0)
T = 100
plt.figure(figsize=(8, 4))
for nombre, tipo, sesgo in [("RNN", "RNN", 0.0), ("LSTM", "LSTM", 0.0),
                            ("LSTM (sesgo=4)", "LSTM", 4.0), ("GRU (sesgo=4)", "GRU", 4.0)]:
    g_t = gradiente_por_paso(RedRecurrente(tipo, sesgo).to(device), T)
    plt.semilogy(range(1, T + 1), g_t, label=nombre)
plt.xlabel("paso de tiempo t (t = 1 es el dígito que debe recordarse)")
plt.ylabel("‖∂L/∂xₜ‖ (escala logarítmica)")
plt.title("Magnitud del gradiente en cada paso de tiempo")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

El extremo derecho de la gráfica corresponde al último paso, el más cercano a la pérdida. Al desplazarse hacia la izquierda, el gradiente recorre la secuencia hacia atrás. En la RNN simple y en la LSTM con inicialización por defecto, su magnitud disminuye en muchos órdenes de magnitud antes de llegar al primer paso, de modo que la red prácticamente no recibe señal sobre la importancia de ese dato. Este fenómeno es el desvanecimiento del gradiente. Con la compuerta inicializada casi abierta, el gradiente se mantiene en un rango similar a lo largo de toda la secuencia.

## Efecto de la longitud de la secuencia

Por último, se entrena cada arquitectura con varias longitudes para observar hasta dónde es capaz de recordar.

In [ ]:
longitudes = [10, 20, 50, 100]
modelos = [("RNN", "RNN", 0.0), ("LSTM (sesgo=4)", "LSTM", 4.0), ("GRU (sesgo=4)", "GRU", 4.0)]
resultados = {nombre: [] for nombre, _, _ in modelos}

for T in longitudes:
    for nombre, tipo, sesgo in modelos:
        _, mejor = entrenar(tipo, T, sesgo_compuerta=sesgo)
        resultados[nombre].append(mejor)

plt.figure(figsize=(7, 4))
for nombre, accs in resultados.items():
    plt.plot(longitudes, accs, marker="o", label=nombre)
plt.axhline(0.1, ls="--", c="gray", lw=1, label="azar (10 %)")
plt.xscale("log")
plt.xticks(longitudes, longitudes)
plt.ylim(0, 1.05)
plt.xlabel("longitud de la secuencia T")
plt.ylabel("mejor exactitud en validación")
plt.title("Exactitud en función de la longitud de la secuencia")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Conclusión

La RNN simple aprende dependencias cortas, pero su capacidad de memoria se degrada rápidamente al aumentar la distancia entre el dato relevante y el momento de la predicción, debido al desvanecimiento del gradiente. Las compuertas de la LSTM y la GRU ofrecen un camino para conservar el estado a lo largo de muchos pasos, aunque su efecto depende de que la red pueda aprovecharlo desde el inicio del entrenamiento; inicializar la compuerta de memoria casi abierta es una práctica habitual para lograrlo. Los resultados pueden variar ligeramente con la semilla aleatoria.